# ⚖️ CourtLLM GRPO Training — Colab-Ready Script

**Runtime:** T4 GPU (free tier) | **Estimated time:** ~45 min (100 steps with early stopping)

Trains an LLM to reduce hallucinations using a multi-agent courtroom environment.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q openenv-core unsloth trl transformers sentence-transformers wikipedia-api httpx matplotlib wandb datasets

## Cell 2: Clone CourtLLM from HuggingFace & Set Up Path

In [ ]:
import os, sys

# Clone the Space repo
if not os.path.exists('/content/CourtLLM_OpenEnv'):
    !git clone https://huggingface.co/spaces/mishatul/CourtLLM_OpenEnv /content/CourtLLM_OpenEnv

# Add root to path so 'from client import ...' and 'from models import ...' work
REPO_ROOT = '/content/CourtLLM_OpenEnv'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repo cloned and path set!')
print('Files:', os.listdir(REPO_ROOT))

## Cell 3: Load Model with Unsloth 4-bit (2x faster, 70% less memory)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Use Qwen2.5-0.5B for fast training — swap to 7B if you have A100
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing=True,
)

print(f"Model loaded: {MODEL_NAME}")

## Cell 4: Connect to CourtLLM Environment

In [ ]:
# Import directly from the cloned repo (avoids pip install issues)
import sys
sys.path.insert(0, '/content/CourtLLM_OpenEnv')

from client import CourtLLMClient
from models import CourtAction

# Live HF Space URL
ENV_URL = "https://mishatul-courtllm-openenv.hf.space"

# Test connection
with CourtLLMClient(ENV_URL) as client:
    health = client.health()
    print(f"Environment status: {health}")

    obs = client.reset()
    print(f"\nCase ID: {obs.case_id}")
    print(f"Query: {obs.plaintiff_query[:100]}...")
    print(f"Evidence sources: {len(obs.evidence_corpus)}")
    print(f"Flagged claims: {len(obs.flagged_claims)}")

## Cell 5: Helper Functions

In [ ]:
import re
from typing import List

def obs_to_prompt(obs, tokenizer) -> str:
    """Convert observation to Defendant system prompt"""
    evidence_str = "\n".join([
        f"[{s['source_id']}] {s['title']}: {s['snippet']}"
        for s in obs.evidence_corpus[:15]
    ])
    claims_str = "\n".join([
        f"- {c['claim_id']}: {c['claim_text']} (Reason: {c['suspicion_reason']})"
        for c in obs.flagged_claims
    ])
    prompt = f"""You are the Defendant in a legal proceeding about factual accuracy.
Respond to the Plaintiff's query with verifiable evidence.

MANDATORY FORMAT:
<claim>
  <statement>Your factual assertion</statement>
  <source_id>EXACT_SOURCE_ID_FROM_CORPUS</source_id>
  <confidence>0.0-1.0</confidence>
</claim>

Evidence Corpus:
{evidence_str}

Plaintiff's Query: {obs.plaintiff_query}

Flagged Claims:
{claims_str}

Your testimony:"""
    return prompt

def parse_defendant_action(completion: str) -> CourtAction:
    """Parse LLM completion into CourtAction"""
    claim_pattern = r'<claim>.*?<statement>(.*?)</statement>.*?<source_id>(.*?)</source_id>.*?<confidence>(.*?)</confidence>.*?</claim>'
    matches = re.findall(claim_pattern, completion, re.DOTALL)
    if not matches:
        return CourtAction(
            action_type="generate_testimony",
            content=completion[:500],
            claim_ids=["claim_000"],
            confidence=0.5,
            source_ids=[]
        )
    statement, source_id, confidence = matches[0]
    try:
        conf = float(confidence.strip())
    except:
        conf = 0.5
    return CourtAction(
        action_type="generate_testimony",
        content=statement.strip(),
        claim_ids=["claim_000"],
        confidence=conf,
        source_ids=[source_id.strip()] if source_id.strip() else []
    )

def generate(model, tokenizer, prompt: str, max_length: int = 256) -> str:
    """Generate completion from model"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("Helper functions loaded!")

## Cell 6: Define Reward Function

In [ ]:
def courtroom_reward_fn(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    """
    For each completion, step the environment and return reward.
    OpenEnv + TRL GRPO pattern.
    """
    rewards = []
    with CourtLLMClient(ENV_URL) as env:
        for completion in completions:
            try:
                obs = env.reset()
                action = parse_defendant_action(completion)
                result = env.step(action)
                rewards.append(result.reward)
            except Exception as e:
                print(f"Error in reward fn: {e}")
                rewards.append(-0.5)
    return rewards

print("Reward function defined!")

## Cell 7: Build Training Dataset (50 episodes — fast)

In [ ]:
from datasets import Dataset

def build_dataset(n_episodes: int = 50, stage: int = 0) -> Dataset:
    """Build prompt dataset from environment resets"""
    prompts = []
    with CourtLLMClient(ENV_URL) as env:
        env.set_stage(stage)
        for i in range(n_episodes):
            obs = env.reset()
            prompt = obs_to_prompt(obs, tokenizer)
            prompts.append({"prompt": prompt})
            if (i + 1) % 10 == 0:
                print(f"Generated {i + 1}/{n_episodes} prompts")
    return Dataset.from_list(prompts)

print("Building training dataset (50 episodes)...")
train_data = build_dataset(n_episodes=50, stage=0)
print(f"Dataset size: {len(train_data)}")

## Cell 8: GRPO Training

**Early stopping enabled at `max_steps=100` (~45 min on T4).**
Full 3-epoch run would take ~34 hours — we use step-based stopping instead.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    output_dir="./courtllm_grpo",
    # --- EARLY STOPPING: caps at 100 steps (~45 min on T4) ---
    max_steps=100,
    num_train_epochs=1,
    # --- Batch config ---
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    # --- GRPO specific ---
    num_generations=4,
    max_prompt_length=768,
    max_completion_length=256,
    temperature=0.8,
    # --- Logging ---
    logging_steps=5,
    save_steps=50,
    report_to="none",   # set to 'wandb' if you want W&B logging
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[courtroom_reward_fn],
    args=config,
    train_dataset=train_data,
)

print("Starting GRPO training (max_steps=100)...")
print("Estimated time: ~45 min on T4 GPU")
trainer.train()
print("Training complete!")

## Cell 9: Plot Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("outputs", exist_ok=True)

log = trainer.state.log_history
steps   = [x["step"]   for x in log if "reward" in x]
rewards = [x["reward"] for x in log if "reward" in x]

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, rewards, label="Episode Reward", linewidth=2, color='#6c63ff')
    plt.axhline(y=0.65, color='g', linestyle='--', label="Target (0.65)")
    plt.xlabel("Training Step", fontsize=12)
    plt.ylabel("Average Episode Reward", fontsize=12)
    plt.title("CourtLLM GRPO Training — Reward Curve", fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/reward_curve_stage1.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Plot saved to outputs/reward_curve_stage1.png")
else:
    print("No reward logs found — run training first.")

## Cell 10: Evaluation — Before/After Conviction Rate

In [ ]:
def eval_conviction_rate(model, tokenizer, env_url: str, n_cases: int = 20) -> float:
    """Evaluate conviction rate on test cases"""
    convictions = 0
    with CourtLLMClient(env_url) as env:
        for i in range(n_cases):
            try:
                obs = env.reset()
                prompt = obs_to_prompt(obs, tokenizer)
                completion = generate(model, tokenizer, prompt)
                action = parse_defendant_action(completion)
                result = env.step(action)
                if result.reward < 0:
                    convictions += 1
                if (i + 1) % 5 == 0:
                    print(f"Evaluated {i + 1}/{n_cases} cases")
            except Exception as e:
                print(f"Error in case {i}: {e}")
                convictions += 1
    return convictions / n_cases

# Quick eval on 20 cases (fast)
print("Evaluating trained model (20 cases)...")
trained_rate = eval_conviction_rate(model, tokenizer, ENV_URL, n_cases=20)
print(f"Trained Conviction Rate: {trained_rate*100:.1f}%")

# Use known baseline from PRD (61% untrained)
baseline_rate = 0.61

labels = ["Baseline\n(untrained)", "CourtLLM\n(GRPO-trained)"]
rates  = [baseline_rate * 100, trained_rate * 100]
colors = ["#e74c3c", "#2ecc71"]

plt.figure(figsize=(8, 6))
bars = plt.bar(labels, rates, color=colors, width=0.5)
plt.ylabel("Conviction Rate (%)", fontsize=12)
plt.title("Hallucination Rate: Before vs After GRPO", fontsize=14, fontweight='bold')
plt.ylim(0, 100)
for bar, rate in zip(bars, rates):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"{rate:.1f}%", ha='center', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("outputs/conviction_rate_drop.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to outputs/conviction_rate_drop.png")

## Cell 11: Save Model & Download Plots

In [ ]:
# Save trained model
model.save_pretrained("courtllm_trained")
tokenizer.save_pretrained("courtllm_trained")
print("Model saved to ./courtllm_trained")

# Download plots to your machine
from google.colab import files
files.download('outputs/reward_curve_stage1.png')
files.download('outputs/conviction_rate_drop.png')
print("\nPlots downloaded! Commit them to your repo.")

# Optional: upload model to HuggingFace Hub
print("\nTo upload model to HuggingFace Hub:")
print("  from huggingface_hub import HfApi")
print("  api = HfApi()")
print("  api.upload_folder(folder_path='courtllm_trained', repo_id='mishatul/courtllm-7b', repo_type='model')")

## ✅ Summary

This notebook:
1. ✅ Loaded Qwen2.5-0.5B with Unsloth 4-bit quantization
2. ✅ Connected to live CourtLLM environment at `mishatul-courtllm-openenv.hf.space`
3. ✅ Trained with GRPO — **100 steps max (~45 min, early stopped)**
4. ✅ Generated reward curve and conviction rate comparison plots
5. ✅ Saved trained model

**Key fix:** Imports directly from cloned repo via `sys.path` — no pip install issues.

**Next:** Commit `outputs/*.png` to your repo, write HF blog post, submit!